In [1]:
import time
import numpy as np
import sklearn
from sklearn.neighbors import KDTree, BallTree
from sklearn.metrics.pairwise import euclidean_distances

import cppimport
import cppimport.import_hook
from extensions.brute_force import distance_matrix, brute_force_query, brute_force_query_radius

class BruteForce(object):

    def __init__(self, points):
        self.points = points
        self.n = self.points.shape[0]
        self.d = self.points.shape[1]
    
    def query(self, X, k=1, return_distance=True, num_threads=4):
        assert X.shape[1] == self.d
        m = X.shape[0]
        dists = np.empty((m,k), dtype=np.float64)
        inds = np.empty((m,k), dtype=np.int64)
        brute_force_query(self.points, X, dists, inds, num_threads)
        if return_distance: return dists, inds
        else: return inds
    
    def query_radius(self, X, r, num_threads=4):
        assert X.shape[1] == self.d
        return brute_force_query_radius(self.points, X, r, num_threads)
        
        

n, d = 10000, 8
points = np.random.uniform(0,1,size=(n,d))

brute_force = BruteForce(points)
tree = KDTree(points)


In [2]:
X = points[:400]

inds = tree.query(X, k=10, return_distance=False)
inds2 = brute_force.query(X, k=10, return_distance=False)
print(np.array_equal(inds, inds2))


True


In [3]:
%%timeit
tree.query(X, k=10, return_distance=False)

6.97 ms ± 47 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [4]:
%%timeit
brute_force.query(X, k=10, return_distance=False, num_threads=1)


195 ms ± 1.1 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [5]:
%%timeit
brute_force.query(X, k=10, return_distance=False, num_threads=2)

101 ms ± 1.4 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [6]:
%%timeit
brute_force.query(X, k=10, return_distance=False, num_threads=4)

50.6 ms ± 119 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [7]:
%%timeit
brute_force.query(X, k=10, return_distance=False, num_threads=8)

27.6 ms ± 845 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [8]:
%%timeit
brute_force.query(X, k=10, return_distance=False, num_threads=12)

24.2 ms ± 1.19 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [9]:
X = points[:150]

counts = tree.query_radius(X, 0.50, count_only=True)

In [10]:
counts

array([ 61,  36,  37,  14,  40,  41, 103,  46,  28,  37,  25,  31,  24,  40,  36,  14,  65,  20,  47,  30,  15,  29,  84,  19,  93,  44,  28,  71,  67,  40,  64,  67,  64,  30,  36,  52,  20,  50,  44,  29,  76, 140,  44,  48,  57,  17,  33,  49,  76,  15,  54,  33,  79,  25,  80,  66,  23,  14,  20,  23,  54,  54,  41,  25,  45,  63, 109,  29,  25,  53,  57,  43,  25,  37,  46,  55, 102,  33,  91,  30, 124,  30,  29,  40,  26,  95,  21, 100,  89,  10,  26,  34,  19,  76,  30,  46,  49,  44,  21,  28,  55,  58,  26, 102,  23,  67,  34,  34,  67, 128, 101,  50,  37,  21,  23,  64,  70,  31,  31,  60,  62,  27,  83,  15,  11,  52,  78,  94,  48,  95,  70,  50,  17,  44,  55,  65,  14,  17, 102,  44,  77, 104,  73,  39,  32,  80,  39,  19,  59,  66])

In [11]:
counts2 = brute_force.query_radius(X, 0.50)

In [12]:
print(np.array_equal(counts, counts2))

True
